**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/lee-2023"], check=True)
    os.chdir("/content/encore/entries/lee-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Lee, Dalley et al. (2023): human against mouse, dandisets 000636 and 000020

Paper: Lee BR, Dalley R, et al. "Signature morphoelectric properties of diverse GABAergic interneurons in the human neocortex." Science 382, eadf6484 (2023), doi:10.1126/science.adf6484.

Human data: [DANDI:000636](https://dandiarchive.org/dandiset/000636/0.241120.0510), version 0.241120.0510, labelled through the paper's code repository (AllenInstitute/human_patchseq_gaba at commit 5eb357a; claim cell-metadata-link, tested in `reproduce_lee_2023.ipynb`).

Mouse data (an input, `data: with_inputs`): the claims file's header says "Mouse comparison data are the VISp Patch-seq GABAergic cells of Gouwens et al. 2020 (ref. 16)". Their recordings are [DANDI:000020](https://dandiarchive.org/dandiset/000020/0.210913.1639) ("Patch-seq recordings from mouse visual cortex"), version 0.210913.1639, its only published version. The NWB files hold no cell type. The per-cell t-types are in the Allen Institute's specimen table `20200711_patchseq_metadata_mouse.csv`; the Allen portal's URL for it has no version, so a public copy pinned at a commit is read (github.com/ewesteinde/patchseq_models at 547d11e, sha256 checked in the helper). It is linked to the files by `ephys_session_id`, which is tested here as the link claim mouse-label-link.

Results, Human versus mouse, paragraph 1: the mouse data are from "the mouse primary visual cortex", so mouse cells are those whose `structure` is VISp with a layer (the table also lists other visual areas). The homologous types are those the Results list: Pvalb 2 is human PVALB WFDC2 and L4-6 PVALB SULF1 and mouse Pvalb Calb1 Sst, Pvalb Reln Itm2a, Pvalb Reln Tac1 and Pvalb Sema3e Kank4; Sst 5 is human SST CALB1 and mouse Sst Calb2 Necab1, Sst Calb2 Pdlim5, Sst Mme Fam114a1, Sst Tac1 Htr1d and Sst Tac1 Tacr3.

| Claim | Tested here |
|---|---|
| mouse-label-link | tested |
| fig6e-species-sag-rheobase-width | tested |
| figs8b-firing-differs-pvalb2-only | tested |
| figs9-species-differences-not-culture | tested |
| fig6c-species-ephys-umap | tested |

Features are computed with the same helper and sweep QC as in `reproduce_lee_2023.ipynb`, for both species. Streaming about 1070 files takes about 30 minutes.

In [ ]:
import time, zlib, resource, warnings, logging, re
import numpy as np, pandas as pd
from scipy import stats
import lee2023 as L
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
KEY = "lee-2023"
def seed(claim, k=0):
    """Seed of a claim's generator (protocol, author step 3); k-th generator adds k."""
    return zlib.crc32(f"{KEY}/{claim}".encode()) + k
SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
T0 = time.time()

## Data

Human: the files of cells of the three human t-types, labelled through the metadata table. Mouse: the label table, the asset list of dandiset 000020, and the files of VISp cells of the nine mouse t-types.

In [ ]:
HUMAN = {"Pvalb 2": ["Inh L2-4 PVALB WFDC2", "Inh L4-6 PVALB SULF1"], "Sst 5": ["Inh L1-3 SST CALB1"]}
MOUSE = {"Pvalb 2": ["Pvalb Calb1 Sst", "Pvalb Reln Itm2a", "Pvalb Reln Tac1", "Pvalb Sema3e Kank4"],
         "Sst 5": ["Sst Calb2 Necab1", "Sst Calb2 Pdlim5", "Sst Mme Fam114a1", "Sst Tac1 Htr1d", "Sst Tac1 Tacr3"]}
meta = L.metadata_table().drop_duplicates()
Ah = L.assets()
hm = meta[meta.Transcriptomic_type.isin(sum(HUMAN.values(), []))].copy()
hm["homolog"] = hm.Transcriptomic_type.map({t_: k for k, v in HUMAN.items() for t_ in v})
hm = hm.merge(Ah, left_on="nwb_session", right_on="session")
M = L.mouse_labels()
Am = L.assets(L.MOUSE_DANDISET, L.MOUSE_VERSION)
M["structure"] = M.structure.astype(str)
M["layer"] = M.structure.str.extract(r"(1|2/3|4|5|6a|6b)$")[0]
M["visp"] = M.structure.str.match(r"^VISp\d")
M["homolog"] = M.ttype.map({t_: k for k, v in MOUSE.items() for t_ in v})
mm = M[M.homolog.notna() & M.visp].merge(Am, on="session")
print("human cells of the homologous types with a file:", hm.groupby(["homolog", "Transcriptomic_type"]).size().to_dict())
print("human M_H_Hodge_cross_species_types of these cells:", hm.groupby("homolog").M_H_Hodge_cross_species_types.value_counts().to_dict())
print("mouse cells of the nine t-types, all areas:", int(M.homolog.notna().sum()), "| in VISp:", int((M.homolog.notna() & M.visp).sum()), "| with a file:", len(mm))
print(mm.groupby(["homolog", "ttype"]).size().to_string())

## Feature extraction

In [ ]:
t = time.time()
Rh = L.all_cell_features(hm.asset_id.tolist(), workers=4)
print(f"human: {len(Rh)} files, {time.time() - t:.0f} s")
t = time.time()
Rm = L.all_cell_features(mm.asset_id.tolist(), workers=4)
print(f"mouse: {len(Rm)} files, {time.time() - t:.0f} s; peak memory of a worker {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")

In [ ]:
fx_paper = L.code_table("LeeDalley_ephys_fx.csv")
FEATS = [c for c in fx_paper.columns if c != "specimen_id" and c not in L.CHIRP and c in Rh.columns and c in Rm.columns]
H = hm.merge(Rh, on="asset_id"); H["species"] = "human"
H["layer_rd_a"] = H.Cortical_layer.astype(str).isin(["2", "3"])
Mo = mm.merge(Rm, on="asset_id"); Mo["species"] = "mouse"
Mo["layer_rd_a"] = Mo.layer == "2/3"
H["human_condition"] = H.condition; Mo["human_condition"] = "mouse"
cols = ["species", "homolog", "layer_rd_a", "human_condition", "n_long_square_qc", "rates"] + FEATS
C = pd.concat([H[cols], Mo[cols]], ignore_index=True)
C = C[C.n_long_square_qc > 0].copy()
NONPOS = ["input_resistance"] + [c for c in FEATS if c.startswith("width_")]
print("non-positive values set missing (as in reproduce_lee_2023.ipynb):", {c: int((C[c] <= 0).sum()) for c in NONPOS if (C[c] <= 0).any()})
for c in NONPOS:
    C.loc[C[c] <= 0, c] = np.nan
print("features computed in both species:", len(FEATS))

## Data check

In [ ]:
print("cells with a QC-passing long square sweep, by species, type and layer reading (a = L2/L3 human, L2/3 mouse):")
print(pd.crosstab([C.homolog, C.species], C.layer_rd_a))
print("human cells by condition:", C[C.species == "human"].groupby("homolog").human_condition.value_counts().to_dict())
print("sampling rates (Hz):"); print(pd.crosstab(C.rates.astype(str), C.species))
print("files with a read error: human", int(Rh.get("error", pd.Series(dtype=object)).notna().sum()), "| mouse", int(Rm.get("error", pd.Series(dtype=object)).notna().sum()))
print("mouse species strings:", Rm.species.value_counts().to_dict())
for c in ["sag", "rheobase_i", "width_rheo", "width_short_square", "avg_rate_hero", "avg_rate_rheo", "fi_fit_slope", "input_resistance"]:
    for sp in ["human", "mouse"]:
        x = C[C.species == sp][c].dropna()
        print(f"  {c:20s} {sp}: n={len(x):4d} min={x.min():.4g} median={x.median():.4g} max={x.max():.4g}")

## mouse-label-link

> The mouse Patch-seq files of dandiset 000020 can be matched to the per-cell t-types of the Allen Institute's specimen table used for Gouwens et al. 2020, and the matched records agree on what both describe.

Matched by `ephys_session_id` (the NWB session id). Reproduced when at least 90% of the table's rows are found in the dandiset, and, over the files read here, the table's sex and genotype agree with the NWB subject's for at least 90% of files.

In [ ]:
found = M.session.isin(Am.session)
print("label rows", len(M), "| found in dandiset 000020:", int(found.sum()), f"({found.mean():.1%})", "| dandiset files without a row:", int((~Am.session.isin(M.session)).sum()), "of", len(Am))
J = mm.merge(Rm[["asset_id", "sex", "genotype", "session_id"]], on="asset_id")
print("session_id in the file equals the file name's:", bool((J.session_id == J.session).all()))
sx = J.biological_sex.astype(str).str.lower().str[0] == J.sex.astype(str).str.lower().str[0]
gt = J.full_genotype.astype(str).str.strip() == J.genotype.astype(str).str.strip()
print(f"over {len(J)} files read: sex agrees {sx.mean():.3f}, genotype agrees {gt.mean():.3f}")
print("examples of disagreement:", J.loc[~(sx & gt), ["session", "biological_sex", "sex", "full_genotype", "genotype"]].head(5).to_dict("records"))
print("RESULT mouse-label-link:", bool(found.mean() >= 0.9 and sx.mean() >= 0.9 and gt.mean() >= 0.9))

**Result: reproduced.** All 4435 rows of the specimen table match a file of dandiset 000020 by session id, and no file lacks a row. Over the 837 files read, the table's sex and genotype agree with the NWB subject's for every file.

## fig6e-species-sag-rheobase-width

> Human and mouse neurons of homologous types Pvalb 2 and Sst 5 differ in sag, rheobase and AP half-width.

Two-sided Mann-Whitney, human against mouse, BH-FDR across the features the claim tests (sag, rheobase_i, width_rheo, width_short_square); BH across all features computed in both species is a variant. Layer readings (a) L2 or L3 in human and L2/3 in mouse, (b) all layers. Human acute and culture cells pooled.

In [ ]:
def species_table(df, feats):
    t_ = L.mannwhitney_table(df, feats, "species", "human", "mouse").set_index("feature")
    return t_.reindex(columns=["n_a", "n_b", "median_a", "median_b", "U", "p", "rbc", "p_fdr"])
F6E = ["sag", "rheobase_i", "width_rheo", "width_short_square"]
RES6E = {}
for rd in ["a", "b"]:
    for hom in ["Pvalb 2", "Sst 5"]:
        d = C[(C.homolog == hom) & (C.layer_rd_a if rd == "a" else True)]
        tt = species_table(d, F6E)
        allf = species_table(d, FEATS)
        tt["p_fdr_all_features"] = allf.loc[F6E, "p_fdr"]
        print(f"--- reading ({rd}), {hom}:"); print(tt[["n_a", "n_b", "median_a", "median_b", "p", "p_fdr", "p_fdr_all_features"]].to_string(float_format=lambda v: f"{v:.4g}"))
        for w in ["width_rheo", "width_short_square"]:
            RES6E[(rd, hom, w)] = bool((tt.loc[["sag", "rheobase_i", w], "p_fdr"] < 1e-4).all())
print(RES6E)
print("RESULT fig6e: every reading holds:", all(RES6E.values()))

**Result: reproduced.** Sag, rheobase and AP width differ between human and mouse at FDR p < 1e-4 in both homologous types, under both layer readings and both width readings (largest FDR p 8.0e-08, width_rheo of Pvalb 2 in L2/3). Human cells have higher sag (median 0.22 against 0.03 in Pvalb 2, 0.41 against 0.14 in Sst 5, L2/3), lower rheobase (90 against 275 pA; 30 against 80 pA) and wider APs (0.52 against 0.42 ms; 0.745 against 0.56 ms at rheobase). Only 38 mouse Pvalb 2 cells are in L2/3.

## figs8b-firing-differs-pvalb2-only

> Average firing rate and f-I slope differ between species for Pvalb 2 but not for Sst 5.

avg_rate at rheobase + 40 pA (first reading) or at rheobase (second reading), and fi_fit_slope; two-sided Mann-Whitney, BH-FDR over the two features of a reading; layer readings as fig6e.

In [ ]:
RES8B = {}
for rd in ["a", "b"]:
    for rate in ["avg_rate_hero", "avg_rate_rheo"]:
        ok = True
        for hom in ["Pvalb 2", "Sst 5"]:
            d = C[(C.homolog == hom) & (C.layer_rd_a if rd == "a" else True)]
            tt = species_table(d, [rate, "fi_fit_slope"])
            print(f"reading ({rd}), {rate}, {hom}:"); print(tt[["n_a", "n_b", "median_a", "median_b", "p", "p_fdr", "rbc"]].to_string(float_format=lambda v: f"{v:.4g}"))
            sig = tt.p_fdr < 0.05
            ok &= bool(sig.all()) if hom == "Pvalb 2" else bool((~sig).all())
        RES8B[(rd, rate)] = ok
print(RES8B)
print("RESULT figs8b: every reading holds:", all(RES8B.values()))

**Result: blocked (methods underspecified).** With the rate at rheobase + 40 pA (the first reading), both layer readings hold: for Pvalb 2 the rate (FDR p 1.0e-06 and 2.4e-16) and the f-I slope (8.3e-06 and 2.8e-22) differ, human lower, and for Sst 5 neither does (rate 0.31 and 0.21, slope 0.15 and 0.21). With the rate at rheobase (the second reading), Sst 5 rates differ (FDR p 7.3e-04 and 5.7e-04, medians 1 and 1 spike/s), so the criterion's "under every reading" fails. The rate at rheobase is near 1 spike/s by construction in both species. The two rate readings give different outcomes, and the claims file's check records the convention that a claim names its sweep readings and is blocked when they disagree, so the claim is blocked. The third verification round agreed. Whether the rate at rheobase can stand for "average firing rate" at all is asked of a reader (questions/lee-2023.yaml).

## figs9-species-differences-not-culture

> Only the Pvalb 2 species difference in AP width is driven mainly by the culture condition; the other species differences in electrophysiology hold for acute and for culture human neurons alike.

For each feature that differs between species in fig6e and figs8b (FDR p < 0.05 there, per type, reading and sweep), human acute against mouse and human culture against mouse, two-sided Mann-Whitney, BH-FDR across those features. Reproduced when every such feature except Pvalb 2 AP width differs at FDR p < 0.05 in the same direction in both comparisons, and Pvalb 2 AP width differs for culture but not for acute human cells.

In [ ]:
RES9 = {}
for rd in ["a", "b"]:
    for w, rate in [("width_rheo", "avg_rate_hero"), ("width_short_square", "avg_rate_rheo")]:
        ok = True
        for hom in ["Pvalb 2", "Sst 5"]:
            d = C[(C.homolog == hom) & (C.layer_rd_a if rd == "a" else True)]
            cand = ["sag", "rheobase_i", w, rate, "fi_fit_slope"]
            base = species_table(d, cand)
            diff = [f for f in cand if base.loc[f, "p_fdr"] < 0.05]
            if not diff:
                print(f"--- reading ({rd}), {w} / {rate}, {hom}: no feature differs between species"); continue
            out = {}
            for cnd in ["acute", "culture"]:
                dd = d[(d.species == "mouse") | (d.human_condition == cnd)]
                out[cnd] = species_table(dd, diff)
            rows = []
            for f in diff:
                sa = np.sign(out["acute"].loc[f, "median_a"] - out["acute"].loc[f, "median_b"]); sc = np.sign(out["culture"].loc[f, "median_a"] - out["culture"].loc[f, "median_b"])
                pa, pc = out["acute"].loc[f, "p_fdr"], out["culture"].loc[f, "p_fdr"]
                if hom == "Pvalb 2" and f == w:
                    good = pc < 0.05 and pa >= 0.05
                else:
                    good = pa < 0.05 and pc < 0.05 and sa == sc
                ok &= bool(good)
                rows.append(dict(feature=f, n_acute=int(out["acute"].loc[f, "n_a"]), n_culture=int(out["culture"].loc[f, "n_a"]), n_mouse=int(out["acute"].loc[f, "n_b"]),
                                 p_fdr_acute=pa, p_fdr_culture=pc, same_direction=sa == sc, as_claimed=good))
            print(f"--- reading ({rd}), {w} / {rate}, {hom}: features differing between species {diff}")
            print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))
        RES9[(rd, w)] = ok
print(RES9)
print("RESULT figs9: every reading holds:", all(RES9.values()))

**Result: blocked (methods underspecified), as it depends on figs8b.** The Pvalb 2 difference in AP width is much stronger with culture human cells (FDR p 1.2e-10 to 1.4e-30) but is also present with acute ones (FDR p 0.011 and 2.9e-05 in L2/3, 3.4e-04 and 4.1e-10 over all layers, same direction), so the part saying it is driven by culture fails as the criterion reads it. The other differences (sag, rheobase, rate at rheobase + 40 pA and f-I slope for Pvalb 2; sag, rheobase and width for Sst 5) hold for acute and culture human cells alike, in the same direction, under the rheobase + 40 pA reading. Under the rheobase-rate reading the rate at rheobase does not (opposite directions in the two comparisons). Only 4 human Sst 5 cells are acute.

This claim depends on figs8b-firing-differs-pvalb2-only for the features it tests. That claim is blocked, so this one is blocked for the same reason under the protocol's rule for dependent claims. Its failing part, the Pvalb 2 width difference with acute cells, does not depend on the rate reading, and without that rule the criterion, which has no partial case, would give not reproduced. The third verification round agreed. Whether "driven primarily by the culture condition" allows a weaker acute difference is asked of a reader (questions/lee-2023.yaml).

## fig6c-species-ephys-umap

> In electrophysiology UMAP space human and mouse neurons separate clearly for both Pvalb 2 and Sst 5.

For each type and layer reading, the human and mouse cells, the features computed in both species z-scored over them (missing values at the median); for each cell, the fraction of its 15 nearest neighbours of its own species, in feature space and in a default umap-learn embedding (claim's seed and four more). Reproduced when the mean is at least 0.9 for both types in both spaces.

In [ ]:
RES6C = {}
for rd in ["a", "b"]:
    for hom in ["Pvalb 2", "Sst 5"]:
        d = C[(C.homolog == hom) & (C.layer_rd_a if rd == "a" else True)]
        Z = L.zscore_impute(d[FEATS]); lab = d.species.values
        sh = pd.Series(lab).value_counts(normalize=True)
        print(f"reading ({rd}) {hom}: species shares {sh.round(3).to_dict()}; own-species fraction expected by chance {float((sh ** 2).sum()):.3f}")
        for off in SEED_OFFSETS:
            emb = L.umap_embed(Z, seed("fig6c-species-ephys-umap") + off)
            for space, Xs in ([("features", Z.values)] if off == 0 else []) + [("umap", emb)]:
                qidx, nb = L.neighbour_fraction(Xs, lab, 15)
                own = np.array([np.mean(n == lab[q]) for q, n in zip(qidx, nb)])
                by = pd.Series(own).groupby(lab[qidx]).mean().round(3).to_dict()
                RES6C[(rd, hom, off, space)] = bool(own.mean() >= 0.9)
                print(f"reading ({rd}) {hom} seed+{off} {space}: cells {len(d)} (human {int((lab == 'human').sum())}), mean own-species fraction {own.mean():.3f} {by}")
main = {k: v for k, v in RES6C.items() if k[2] == 0}
print("RESULT fig6c, claim's seed: every reading holds:", all(main.values()), "| over five seeds:", all(RES6C.values()))

**Result: not reproduced.** No homologous type reaches a mean own-species fraction of 0.9 in both spaces under either layer reading. Pvalb 2 misses narrowly in feature space (0.895 in L2/3, 0.893 over all layers) and passes in UMAP (0.914 to 0.949 over the five seeds and both layer readings). Sst 5 fails in both spaces (L2/3: 0.805 in feature space, 0.823 to 0.850 in UMAP; all layers: 0.895 and 0.875 to 0.888). Human Sst 5 cells have 0.38 to 0.69 human neighbours, above their share of the cells (0.27 and 0.13) but far from separate. The means are set mostly by the larger species group; the own-species fractions expected by chance are 0.59 to 0.78.

## Summary

| Claim | Outcome |
|---|---|
| mouse-label-link | reproduced |
| fig6e-species-sag-rheobase-width | reproduced |
| figs8b-firing-differs-pvalb2-only | blocked (methods underspecified) |
| figs9-species-differences-not-culture | blocked (methods underspecified; depends on figs8b) |
| fig6c-species-ephys-umap | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The mouse features are computed here with the same IPFX code and sweep QC as the human ones, not taken from Gouwens et al. 2020. 65 of the 84 features are computed (the others need unreleased code). The mouse cells are those of VISp in the specimen table; the UMAP claim is judged by the neighbour measure the claims file fixes. UMAP embeddings differ between runs on different machines under the same seed and lock file (umap-learn and its nearest-neighbour search), so UMAP numbers can differ in another run of this notebook. The fig. S9 legend is missing, so its test is read from the Results.

What was found about the data. The mouse files carry no t-type; the labels come from the Allen Institute's specimen table, whose portal URL has no version, read from a pinned public copy. The table and the dandiset match one to one by session id.

In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")